# Model Difusi (DDPM) dari Nol pada Fashion-MNIST

**Buku:** *Deep Learning dengan Python* — Bab 15, Model Difusi

Proses maju menambahkan derau Gaussian sedikit demi sedikit. Jaringan U-Net belajar menebak derau tersebut. Pembangkitan dimulai dari derau murni, lalu derau dikurangi langkah demi langkah.

**Lingkungan:** Google Colab (GPU disarankan), Python 3.10+. Kode bersifat *backend-agnostik* (TensorFlow, JAX, atau PyTorch).

**Perkiraan waktu (GPU T4):** ±25 menit untuk 30 epoch

**Isi notebook:**
- Jadwal derau linear β₁…β_T (T = 1000) dan rumus bentuk tertutup $x_t = \sqrt{\bar\alpha_t}\,x_0 + \sqrt{1-\bar\alpha_t}\,\varepsilon$.
- U-Net kecil dengan embedding waktu sinusoidal, GroupNorm, dan blok residual.
- Pelatihan dengan `PyDataset` (penambahan derau di sisi data) dan loss MSE pada derau.
- *Sampling* DDPM (1000 langkah) vs DDIM (50 langkah, deterministik).

In [ ]:
import os
# Pilih backend: "tensorflow", "jax", atau "torch"
os.environ["KERAS_BACKEND"] = "tensorflow"

import keras
print("Keras", keras.__version__, "| backend:", keras.backend.backend())

## 1. Data dan jadwal derau

In [ ]:
import math, time
import numpy as np
import matplotlib.pyplot as plt
from keras import layers, ops

(x_latih, _), _ = keras.datasets.fashion_mnist.load_data()
x_latih = np.pad(x_latih, ((0, 0), (2, 2), (2, 2)))                 # 28 -> 32 agar mudah di-downsample
x_latih = (x_latih[..., None].astype("float32") / 127.5) - 1.0       # skala [-1, 1]

T = 1000
beta = np.linspace(1e-4, 0.02, T, dtype="float32")
alpha = 1.0 - beta
alpha_bar = np.cumprod(alpha).astype("float32")

def tambah_derau(x0, t, eps):
    ab = alpha_bar[t].reshape(-1, 1, 1, 1)
    return np.sqrt(ab) * x0 + np.sqrt(1 - ab) * eps

langkah_t = [0, 50, 100, 200, 400, 600, 800, 999]
rng = np.random.default_rng(0)
contoh = tambah_derau(np.repeat(x_latih[:1], len(langkah_t), 0), np.array(langkah_t),
                      rng.standard_normal((len(langkah_t), 32, 32, 1)).astype("float32"))
fig, ax = plt.subplots(1, len(langkah_t), figsize=(14, 2.2))
for a, g, t in zip(ax, contoh, langkah_t):
    a.imshow(g[..., 0], cmap="gray", vmin=-1, vmax=1); a.set_title(f"t = {t}"); a.axis("off")
plt.suptitle("Proses maju"); plt.show()
plt.figure(figsize=(5, 3)); plt.plot(alpha_bar); plt.xlabel("t"); plt.ylabel(r"$\bar\alpha_t$ (sisa sinyal)"); plt.show()

## 2. Data latih: pasangan $((x_t, t),\ \varepsilon)$
Derau dan $t$ diundi ulang setiap batch, sehingga model melihat semua tingkat derau.

In [ ]:
class DataDifusi(keras.utils.PyDataset):
    def __init__(self, x, batch=128, **kwargs):
        super().__init__(**kwargs)
        self.x, self.batch = x, batch
        self.rng = np.random.default_rng(1)
        self.urutan = self.rng.permutation(len(x))

    def __len__(self):
        return len(self.x) // self.batch

    def __getitem__(self, i):
        x0 = self.x[self.urutan[i * self.batch:(i + 1) * self.batch]]
        t = self.rng.integers(0, T, size=(len(x0), 1))
        eps = self.rng.standard_normal(x0.shape).astype("float32")
        return (tambah_derau(x0, t[:, 0], eps), t), eps

    def on_epoch_end(self):
        self.urutan = self.rng.permutation(len(self.x))

## 3. U-Net dengan embedding waktu

In [ ]:
class EmbeddingWaktu(layers.Layer):
    """t (bilangan bulat) -> vektor sinusoidal, seperti positional encoding Transformer."""
    def __init__(self, dim, **kwargs):
        super().__init__(**kwargs)
        self.dim = dim

    def call(self, t):
        setengah = self.dim // 2
        frek = ops.exp(-math.log(10000.0) * ops.arange(setengah, dtype="float32") / setengah)
        arg = ops.cast(t, "float32") * ops.expand_dims(frek, 0)        # (B, 1) * (1, setengah)
        return ops.concatenate([ops.sin(arg), ops.cos(arg)], axis=-1)

    def get_config(self):
        return {**super().get_config(), "dim": self.dim}


def blok_res(x, emb, f):
    h = layers.GroupNormalization(groups=8)(x)
    h = layers.Activation("swish")(h)
    h = layers.Conv2D(f, 3, padding="same")(h)
    waktu = layers.Reshape((1, 1, f))(layers.Dense(f)(emb))          # sisipkan informasi t
    h = layers.add([h, waktu])
    h = layers.GroupNormalization(groups=8)(h)
    h = layers.Activation("swish")(h)
    h = layers.Conv2D(f, 3, padding="same")(h)
    if x.shape[-1] != f:
        x = layers.Conv2D(f, 1)(x)
    return layers.add([x, h])


def unet_difusi(filter=(64, 128, 256), dim_waktu=128):
    gambar = keras.Input(shape=(32, 32, 1), name="x_t")
    t = keras.Input(shape=(1,), dtype="int32", name="t")
    emb = EmbeddingWaktu(dim_waktu)(t)
    emb = layers.Dense(dim_waktu * 2, activation="swish")(emb)
    emb = layers.Dense(dim_waktu * 2, activation="swish")(emb)

    x = layers.Conv2D(filter[0], 3, padding="same")(gambar)
    lewatan = []
    for i, f in enumerate(filter):                              # 32 -> 16 -> 8
        x = blok_res(x, emb, f)
        lewatan.append(x)
        if i < len(filter) - 1:
            x = layers.Conv2D(f, 3, strides=2, padding="same")(x)
    x = blok_res(x, emb, filter[-1])                             # bottleneck 8x8
    for i, (f, s) in enumerate(zip(reversed(filter), reversed(lewatan))):
        x = layers.Concatenate()([x, s])
        x = blok_res(x, emb, f)
        if i < len(filter) - 1:
            x = layers.UpSampling2D(2)(x)
    x = layers.GroupNormalization(groups=8)(x)
    x = layers.Activation("swish")(x)
    keluaran = layers.Conv2D(1, 3, padding="same", kernel_initializer="zeros")(x)   # tebakan derau
    return keras.Model([gambar, t], keluaran)

model = unet_difusi()
print(f"Parameter: {model.count_params():,}")

## 4. Pelatihan
Optimizer dengan EMA bawaan Keras (`use_ema=True`). Di akhir `fit`, bobot EMA otomatis dipasang ke model.

In [ ]:
model.compile(optimizer=keras.optimizers.AdamW(learning_rate=1e-3, weight_decay=1e-4,
                                               use_ema=True, ema_momentum=0.999),
              loss="mse")
riwayat = model.fit(DataDifusi(x_latih), epochs=30)
model.save("ddpm_fashion.keras")
plt.plot(riwayat.history["loss"]); plt.xlabel("Epoch"); plt.ylabel("MSE derau"); plt.show()

## 5. Pembangkitan: DDPM (stokastik, 1000 langkah)
$x_{t-1} = \frac{1}{\sqrt{\alpha_t}}\Big(x_t - \frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\,\hat\varepsilon\Big) + \sqrt{\beta_t}\,z$

In [ ]:
def prediksi_derau(x, t):
    t_batch = np.full((len(x), 1), t, dtype="int32")
    return keras.ops.convert_to_numpy(model([x, t_batch], training=False))

def sampel_ddpm(n=32, seed=0, simpan_setiap=None):
    rng = np.random.default_rng(seed)
    x = rng.standard_normal((n, 32, 32, 1)).astype("float32")
    jejak = []
    for t in reversed(range(T)):
        eps = prediksi_derau(x, t)
        x = (x - beta[t] / np.sqrt(1 - alpha_bar[t]) * eps) / np.sqrt(alpha[t])
        if t > 0:
            x += np.sqrt(beta[t]) * rng.standard_normal(x.shape).astype("float32")
        if simpan_setiap and t % simpan_setiap == 0:
            jejak.append(x[0].copy())
    return np.clip(x, -1, 1), jejak

def tampilkan(gambar, judul, kolom=8):
    baris = int(math.ceil(len(gambar) / kolom))
    fig, ax = plt.subplots(baris, kolom, figsize=(kolom * 1.4, baris * 1.4))
    for a, g in zip(np.ravel(ax), gambar):
        a.imshow(g[..., 0], cmap="gray", vmin=-1, vmax=1)
    for a in np.ravel(ax):
        a.axis("off")
    plt.suptitle(judul); plt.show()

t0 = time.time()
hasil_ddpm, jejak = sampel_ddpm(32, simpan_setiap=125)
print(f"DDPM: {time.time() - t0:.0f} detik")
tampilkan(hasil_ddpm, "DDPM, 1000 langkah")
tampilkan(np.array(jejak), "Proses balik satu sampel (t = 875, 750, ..., 0)")

## 6. DDIM: 50 langkah, deterministik
DDIM memakai model yang **sama** tanpa pelatihan ulang, tetapi melompati sebagian besar langkah waktu.

In [ ]:
def sampel_ddim(n=32, langkah=50, seed=0):
    rng = np.random.default_rng(seed)
    x = rng.standard_normal((n, 32, 32, 1)).astype("float32")
    jadwal_t = np.linspace(T - 1, 0, langkah).astype(int)
    for i, t in enumerate(jadwal_t):
        eps = prediksi_derau(x, t)
        x0_tebakan = (x - np.sqrt(1 - alpha_bar[t]) * eps) / np.sqrt(alpha_bar[t])
        x0_tebakan = np.clip(x0_tebakan, -1, 1)
        if i == len(jadwal_t) - 1:
            return x0_tebakan
        ab_berikut = alpha_bar[jadwal_t[i + 1]]
        x = np.sqrt(ab_berikut) * x0_tebakan + np.sqrt(1 - ab_berikut) * eps

t0 = time.time()
hasil_ddim = sampel_ddim(32)
print(f"DDIM: {time.time() - t0:.1f} detik")
tampilkan(hasil_ddim, "DDIM, 50 langkah (seed sama dengan DDPM)")

**Eksperimen:**
1. Ganti jadwal linear dengan jadwal kosinus (Nichol & Dhariwal, 2021), lalu bandingkan kualitas pada jumlah epoch yang sama.
2. Kurangi DDIM menjadi 10 atau 20 langkah. Di titik mana kualitasnya runtuh?
3. Tambahkan label kelas sebagai masukan ketiga (embedding yang dijumlahkan ke `emb`) agar model bisa diminta membangkitkan kelas tertentu. Ini adalah dasar *classifier-free guidance*.